# Solar activity and inflation: falsification-oriented mechanism study

Public project: [CastaliaInstitute/solar-revolutions](https://github.com/CastaliaInstitute/solar-revolutions)  
Public page: [castaliainstitute.github.io/solar-revolutions](https://castaliainstitute.github.io/solar-revolutions/)

## Study goal

**Goal: try to disprove the hypothesis** that SILSO solar activity meaningfully predicts inflation through a plausible mechanism. The analysis is designed to look for ways the apparent signal could fail: placebo cycles, conventional controls, lagged inflation, historical-period instability, and weak mechanism candidates.

## Scientific posture

The starting observation is narrow: in the current catalog-wide one-input screen, annual SILSO solar activity predicts Shiller annual inflation better than other single inputs in held-out years. This notebook treats that as a suspicious exploratory finding, not as confirmation.

The burden of proof is on the solar-inflation hypothesis. A visually or statistically interesting one-input result is not enough.


## 1. Formal hypotheses and falsification criteria

### Null hypothesis, H0

SILSO solar activity does **not** meaningfully predict inflation beyond chance, autocorrelation, historical regime shifts, or conventional macroeconomic predictors. Any apparent relationship is an artifact of time structure, sample choice, omitted variables, or coincidental cycle alignment.

### Alternative hypothesis, H1

SILSO solar activity meaningfully predicts inflation because solar or geomagnetic activity affects one or more intermediate systems, such as agriculture, food prices, energy systems, communications, logistics, or infrastructure. These disruptions then contribute to measurable price pressure.

### Candidate causal pathway

`solar activity -> geomagnetic/radiation/weather/infrastructure disturbance -> food, energy, logistics, or production stress -> inflation`

### What would weaken or disprove H1?

H1 is weakened if any of the following occur:

1. Shifted SILSO or generic 9-14 year sine waves predict inflation about as well as real SILSO.
2. SILSO loses held-out predictive value after lagged inflation and conventional controls.
3. The relationship appears only in one historical period or monetary regime.
4. Mechanism candidates such as food prices, crop yields, energy/weather proxies, or rates do not align with both SILSO and inflation.
5. Adding inflation does not reduce any solar/revolution association in mediation-style screens.

### What would keep H1 alive?

H1 remains worth studying only if real SILSO beats placebo cycles, adds out-of-sample value after conventional controls, shows stable lag structure, and connects to plausible intermediate variables.


## 2. Setup

Run this section first. In Colab, the notebook clones the public repository if needed.


In [ ]:
#@title Install notebook dependencies { display-mode: "form" }
INSTALL_PACKAGES = False

if INSTALL_PACKAGES:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pandas>=2.2,<3', 'numpy>=1.26,<3', 'matplotlib>=3.8,<4', 'scipy>=1.11,<2'], check=True)


In [ ]:
#@title Locate or clone public repository { display-mode: "form" }
import json
import os
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_URL = 'https://github.com/CastaliaInstitute/solar-revolutions.git'

def running_in_colab():
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

def api_root_for(candidate: Path):
    for api in [candidate / 'api', candidate / 'dashboard' / 'public' / 'api']:
        if (api / 'input-data' / 'climate_solar_activity.json').exists():
            return api
    return None

def find_repo():
    env_root = os.environ.get('SOLAR_REPO_ROOT') or os.environ.get('APOCALYPSO_REPO_ROOT')
    candidates = []
    if env_root:
        candidates.append(Path(env_root).expanduser())
    candidates.extend([Path.cwd(), *Path.cwd().parents, Path('/content/solar-revolutions')])
    for candidate in candidates:
        api = api_root_for(candidate)
        if api is not None:
            return candidate, api
    if running_in_colab():
        target = Path('/content/solar-revolutions')
        if not target.exists():
            subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(target)], check=True)
        api = api_root_for(target)
        if api is not None:
            return target, api
    raise FileNotFoundError('Could not find public API artifacts. Set SOLAR_REPO_ROOT to the solar-revolutions checkout.')

ROOT, API = find_repo()
INPUT_DATA = API / 'input-data'
RED_API = API / 'red'
print('Repo root:', ROOT)
print('API:', API)


## 3. Load solar, inflation, and mechanism-candidate inputs


In [ ]:
#@title Load annualized input series { display-mode: "form" }
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def parse_year(date):
    try:
        return int(str(date)[:4])
    except Exception:
        return None

def annualize(series_id):
    path = INPUT_DATA / f'{series_id.replace('.', '_')}.json'
    data = load_json(path)
    buckets = {}
    for point in data.get('points', []):
        year = point.get('year') if isinstance(point.get('year'), int) else parse_year(point.get('date'))
        try:
            value = float(point.get('value'))
        except Exception:
            continue
        if year is None or not np.isfinite(value):
            continue
        buckets.setdefault(year, []).append(value)
    return pd.Series({year: float(np.mean(values)) for year, values in buckets.items()}, name=series_id).sort_index()

series_ids = {
    'silso': 'climate.solar_activity',
    'inflation_shiller': 'inflation.shiller_annual_pct',
    'cpi_shiller': 'inflation.shiller_cpi',
    'food_price': 'food.long_term_food_price_index',
    'food_anomaly': 'food.food_price_anomaly',
    'cereal_yield': 'agriculture.wdi_cereal_yield',
    'food_production': 'agriculture.wdi_food_production_index',
    'temperature': 'climate.temperature_anomaly',
    'short_rate': 'finance.jst_short_rate_mean',
    'long_rate': 'rates.shiller_long_rate_gs10',
    'gdp_growth': 'macro.wdi_gdp_growth',
    'revolutions': 'revolutions.event_count',
}

loaded = {}
for label, sid in series_ids.items():
    path = INPUT_DATA / f'{sid.replace('.', '_')}.json'
    if path.exists():
        loaded[label] = annualize(sid)
    else:
        print('Missing:', sid)
df = pd.concat(loaded.values(), axis=1)
df.columns = list(loaded.keys())
df = df.sort_index()
df.tail()


## 4. Baseline SILSO-inflation signal

This reproduces the basic annual relationship using the public inputs. The stronger test is not same-year correlation; it is whether the relationship survives lags, controls, splits, and placebo solar cycles.


In [ ]:
#@title Same-year and lagged correlations { display-mode: "form" }
def corr_at_lag(data, x, y, lag):
    tmp = pd.DataFrame({'x': data[x].shift(lag), 'y': data[y]}).dropna()
    if len(tmp) < 10:
        return {'lag': lag, 'n': len(tmp), 'r': np.nan}
    return {'lag': lag, 'n': len(tmp), 'r': tmp['x'].corr(tmp['y'])}

lag_rows = [corr_at_lag(df, 'silso', 'inflation_shiller', lag) for lag in range(-11, 12)]
lag_df = pd.DataFrame(lag_rows)
lag_df.sort_values('r', key=lambda s: s.abs(), ascending=False).head(12)


In [ ]:
#@title Plot annual SILSO and Shiller inflation { display-mode: "form" }
plot_df = df[['silso', 'inflation_shiller']].dropna().copy()
plot_df['silso_z'] = (plot_df['silso'] - plot_df['silso'].mean()) / plot_df['silso'].std()
plot_df['inflation_z'] = (plot_df['inflation_shiller'] - plot_df['inflation_shiller'].mean()) / plot_df['inflation_shiller'].std()
ax = plot_df[['silso_z', 'inflation_z']].plot(figsize=(12, 4), lw=1.4)
ax.axhline(0, color='gray', lw=0.8)
ax.set_title('Standardized annual SILSO and Shiller inflation')
ax.set_ylabel('z-score')
plt.show()


## 5. First placebo-cycle tests

A solar-specific result should beat shifted solar cycles and generic 11-year sine waves. If placebo cycles perform similarly, the finding is probably time-structure leakage.


In [ ]:
#@title Compare real SILSO to shifted and synthetic cycles { display-mode: "form" }
base = df[['silso', 'inflation_shiller']].dropna().copy()
real_r = base['silso'].corr(base['inflation_shiller'])
shift_rows = []
for shift in range(1, len(base)):
    shifted = np.roll(base['silso'].to_numpy(), shift)
    shift_rows.append({'placebo': f'circular_shift_{shift}', 'r': np.corrcoef(shifted, base['inflation_shiller'])[0, 1]})
for period in [9, 10, 11, 12, 13]:
    years = base.index.to_numpy()
    wave = np.sin(2 * np.pi * (years - years.min()) / period)
    shift_rows.append({'placebo': f'sine_{period}y', 'r': np.corrcoef(wave, base['inflation_shiller'])[0, 1]})
placebo_df = pd.DataFrame(shift_rows)
p_two_sided = (placebo_df['r'].abs() >= abs(real_r)).mean()
print('Real same-year r:', real_r)
print('Placebo p, two-sided:', p_two_sided)
placebo_df.assign(abs_r=lambda x: x['r'].abs()).sort_values('abs_r', ascending=False).head(12)


## 6. Catalog-wide inflation predictor screen

Before treating the SILSO-inflation relationship as special, compare SILSO against every other available input. This screen treats Shiller annual inflation as the outcome and fits one predictor at a time.

Interpretation rule: positive held-out R² means the input improves prediction over a training-period mean baseline in the held-out period. High held-out correlation with negative R² is weaker because the model still predicts worse than a simple baseline.


In [ ]:
#@title Rank all non-inflation inputs predicting Shiller annual inflation { display-mode: "form" }
catalog = load_json(API / "input-series.json")["series"]
YEARS = list(range(1900, 2015))

def ridge_one_predict(x_train, y_train, x_test, alpha=5.0):
    x_train = np.asarray(x_train, dtype=float).reshape(-1, 1)
    x_test = np.asarray(x_test, dtype=float).reshape(-1, 1)
    y_train = np.asarray(y_train, dtype=float)
    mean = x_train.mean(axis=0)
    sd = x_train.std(axis=0)
    sd[sd == 0] = 1.0
    X = (x_train - mean) / sd
    Xt = (x_test - mean) / sd
    D = np.column_stack([np.ones(len(X)), X])
    T = np.column_stack([np.ones(len(Xt)), Xt])
    penalty = np.eye(D.shape[1]) * alpha
    penalty[0, 0] = 0.0
    beta = np.linalg.pinv(D.T @ D + penalty) @ D.T @ y_train
    return T @ beta

def regression_metrics(y, pred):
    y = np.asarray(y, dtype=float)
    pred = np.asarray(pred, dtype=float)
    err = pred - y
    ss_res = float(np.sum(err ** 2))
    ss_tot = float(np.sum((y - y.mean()) ** 2))
    return {
        "rmse": float(np.sqrt(np.mean(err ** 2))),
        "mae": float(np.mean(np.abs(err))),
        "r2": 1 - ss_res / ss_tot if ss_tot else np.nan,
        "pearson": float(np.corrcoef(y, pred)[0, 1]) if len(y) >= 3 and y.std() > 0 and pred.std() > 0 else np.nan,
    }

inflation = annualize("inflation.shiller_annual_pct")
split = int(len(YEARS) * 0.7)
train_years = YEARS[:split]
test_years = YEARS[split:]
rows = []
for meta in catalog:
    sid = meta.get("id", "")
    if not sid or sid.startswith("inflation."):
        continue
    path = INPUT_DATA / f"{sid.replace(".", "_")}.json"
    if not path.exists():
        continue
    values = annualize(sid)
    train = [year for year in train_years if year in values and year in inflation]
    test = [year for year in test_years if year in values and year in inflation]
    if len(train) < 10 or len(test) < 3:
        continue
    y_train = [inflation[year] for year in train]
    x_train = [values[year] for year in train]
    y_test = [inflation[year] for year in test]
    x_test = [values[year] for year in test]
    pred = ridge_one_predict(x_train, y_train, x_test)
    mean_pred = np.repeat(float(np.mean(y_train)), len(y_test))
    model = regression_metrics(y_test, pred)
    baseline = regression_metrics(y_test, mean_pred)
    overlap = [year for year in YEARS if year in values and year in inflation]
    same_year_r = np.nan
    if len(overlap) >= 10:
        a = np.asarray([values[year] for year in overlap], dtype=float)
        b = np.asarray([inflation[year] for year in overlap], dtype=float)
        if a.std() > 0 and b.std() > 0:
            same_year_r = float(np.corrcoef(a, b)[0, 1])
    rows.append({
        "id": sid,
        "name": meta.get("name", sid),
        "source": meta.get("source"),
        "train_n": len(train),
        "test_n": len(test),
        "heldout_r2": model["r2"],
        "heldout_pearson": model["pearson"],
        "delta_r2_vs_mean": model["r2"] - baseline["r2"],
        "same_year_r": same_year_r,
        "rmse": model["rmse"],
    })
inflation_predictors = pd.DataFrame(rows).sort_values(["heldout_r2", "heldout_pearson"], ascending=[False, False])
inflation_predictors.head(20)


### Reading the inflation-predictor screen

If SILSO remains near the top after this comparison, it is an interesting signal. But the comparison also identifies conventional controls that should be included before making any mechanism claim: debt, rates, conflict, market drawdowns, food/agriculture, and crisis measures. Revolution variables in this table are not clean causal predictors of inflation; they are more likely shared-crisis indicators or reverse/entangled relationships.


## 7. Does SILSO add value after conventional inflation predictors?

The previous table shows SILSO is the strongest one-input predictor in this screen. The stricter question is whether SILSO still adds held-out predictive value after lagged inflation, debt, rates, conflict, markets, and food/agriculture controls.


In [ ]:
#@title Incremental held-out R² from adding SILSO to controls { display-mode: "form" }
def standardize_train_apply(x_train, x_test):
    mean = np.nanmean(x_train, axis=0)
    sd = np.nanstd(x_train, axis=0)
    sd[~np.isfinite(sd) | (sd == 0)] = 1.0
    return (x_train - mean) / sd, (x_test - mean) / sd

def ridge_multi_predict(frame, y_col, x_cols, train_years, test_years, alpha=10.0):
    train = frame.loc[frame.index.intersection(train_years), [y_col, *x_cols]].dropna()
    test = frame.loc[frame.index.intersection(test_years), [y_col, *x_cols]].dropna()
    if len(train) < 10 or len(test) < 3:
        return None
    y_train = train[y_col].to_numpy(dtype=float)
    y_test = test[y_col].to_numpy(dtype=float)
    x_train = train[x_cols].to_numpy(dtype=float)
    x_test = test[x_cols].to_numpy(dtype=float)
    x_train, x_test = standardize_train_apply(x_train, x_test)
    D = np.column_stack([np.ones(len(x_train)), x_train])
    T = np.column_stack([np.ones(len(x_test)), x_test])
    penalty = np.eye(D.shape[1]) * alpha
    penalty[0, 0] = 0.0
    beta = np.linalg.pinv(D.T @ D + penalty) @ D.T @ y_train
    pred = T @ beta
    out = regression_metrics(y_test, pred)
    out.update({"train_n": len(train), "test_n": len(test)})
    return out

control_df = df.copy()
control_df["inflation_lag1"] = control_df["inflation_shiller"].shift(1)
candidate_controls = {
    "lag_only": ["inflation_lag1"],
    "debt_rates": ["inflation_lag1", "short_rate", "long_rate"],
    "food_agriculture": ["inflation_lag1", "food_price", "food_anomaly", "food_production", "cereal_yield"],
    "macro_conflict_market": ["inflation_lag1", "gdp_growth", "short_rate", "long_rate", "market_shiller_real_drawdown", "conflict_state_based_conflict_deaths"],
}
# Keep only controls that exist in this public checkout.
candidate_controls = {name: [col for col in cols if col in control_df.columns] for name, cols in candidate_controls.items()}
rows = []
for name, cols in candidate_controls.items():
    base = ridge_multi_predict(control_df, "inflation_shiller", cols, train_years, test_years)
    with_silso = ridge_multi_predict(control_df, "inflation_shiller", [*cols, "silso"], train_years, test_years)
    if base is None or with_silso is None:
        continue
    rows.append({
        "control_set": name,
        "controls": ", ".join(cols),
        "base_r2": base["r2"],
        "with_silso_r2": with_silso["r2"],
        "delta_r2_from_silso": with_silso["r2"] - base["r2"],
        "with_silso_pearson": with_silso["pearson"],
        "test_n": with_silso["test_n"],
    })
incremental_silso = pd.DataFrame(rows).sort_values("delta_r2_from_silso", ascending=False)
incremental_silso


## 8. Period splits

A robust mechanism should not depend entirely on one monetary regime. These splits are rough screens, not final inference.


In [ ]:
#@title SILSO-inflation relationship by historical period { display-mode: "form" }
periods = {
    "1900-1939": range(1900, 1940),
    "1940-1979": range(1940, 1980),
    "1980-2014": range(1980, 2015),
    "post_1950": range(1950, 2015),
}
period_rows = []
for name, years in periods.items():
    tmp = df.loc[df.index.intersection(list(years)), ["silso", "inflation_shiller"]].dropna()
    if len(tmp) < 10:
        continue
    period_rows.append({
        "period": name,
        "n": len(tmp),
        "same_year_r": tmp["silso"].corr(tmp["inflation_shiller"]),
        "inflation_mean": tmp["inflation_shiller"].mean(),
        "inflation_sd": tmp["inflation_shiller"].std(),
    })
pd.DataFrame(period_rows)


## 9. Does real SILSO beat placebo cycles in prediction?

This is the key skeptical test. We compare the held-out R² of real SILSO against circularly shifted SILSO and generic sine waves. If many placebo cycles do as well or better, the result is not solar-specific.


In [ ]:
#@title Held-out prediction: real SILSO vs shifted/synthetic cycles { display-mode: "form" }
base = df[["silso", "inflation_shiller"]].dropna().loc[1900:2014].copy()
years = base.index.to_list()
split = int(len(years) * 0.7)
tr_years = years[:split]
te_years = years[split:]

def one_predictor_r2(series):
    frame = pd.DataFrame({"x": series, "inflation": base["inflation_shiller"]}, index=base.index)
    tr = frame.loc[tr_years].dropna()
    te = frame.loc[te_years].dropna()
    pred = ridge_one_predict(tr["x"], tr["inflation"], te["x"])
    return regression_metrics(te["inflation"], pred)["r2"]

real_r2 = one_predictor_r2(base["silso"])
placebo_rows = []
values = base["silso"].to_numpy()
for shift in range(1, len(values)):
    placebo_rows.append({"placebo": f"circular_shift_{shift}", "r2": one_predictor_r2(pd.Series(np.roll(values, shift), index=base.index))})
for period in [8, 9, 10, 11, 12, 13, 14]:
    wave = np.sin(2 * np.pi * (base.index.to_numpy() - base.index.min()) / period)
    placebo_rows.append({"placebo": f"sine_{period}y", "r2": one_predictor_r2(pd.Series(wave, index=base.index))})
placebo_pred = pd.DataFrame(placebo_rows).sort_values("r2", ascending=False)
print("Real SILSO held-out R²:", real_r2)
print("Share of placebo cycles with R² >= real:", (placebo_pred["r2"] >= real_r2).mean())
placebo_pred.head(15)


## 10. Mechanism candidates

If the inflation signal is real, food, agriculture, energy/weather, or rates should show related structure. This section screens available mechanism candidates.


In [ ]:
#@title Correlate SILSO and inflation with mechanism candidates { display-mode: "form" }
candidate_cols = [c for c in df.columns if c not in ['silso', 'inflation_shiller']]
rows = []
for col in candidate_cols:
    tmp = df[['silso', 'inflation_shiller', col]].dropna()
    if len(tmp) < 10:
        continue
    rows.append({
        'candidate': col,
        'n': len(tmp),
        'r_silso_candidate': tmp['silso'].corr(tmp[col]),
        'r_inflation_candidate': tmp['inflation_shiller'].corr(tmp[col]),
    })
pd.DataFrame(rows).assign(score=lambda x: x['r_silso_candidate'].abs() + x['r_inflation_candidate'].abs()).sort_values('score', ascending=False)


## 11. Simple control models

These are deliberately simple OLS screens. The key question is whether SILSO adds anything after lagged inflation and obvious historical controls.


In [ ]:
#@title OLS screens with lagged inflation and controls { display-mode: "form" }
def ols_fit(frame, y_col, x_cols):
    tmp = frame[[y_col, *x_cols]].dropna()
    y = tmp[y_col].to_numpy(dtype=float)
    X = tmp[x_cols].to_numpy(dtype=float)
    X = np.column_stack([np.ones(len(X)), X])
    beta = np.linalg.pinv(X.T @ X) @ X.T @ y
    pred = X @ beta
    ss_res = np.sum((y - pred) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    return {'n': len(tmp), 'r2': 1 - ss_res / ss_tot if ss_tot else np.nan, **{f'beta_{name}': beta[i+1] for i, name in enumerate(x_cols)}}

model_df = df.copy()
model_df['inflation_lag1'] = model_df['inflation_shiller'].shift(1)
specs = {
    'lag_only': ['inflation_lag1'],
    'lag_plus_silso': ['inflation_lag1', 'silso'],
    'lag_silso_rates': [c for c in ['inflation_lag1', 'silso', 'short_rate', 'long_rate'] if c in model_df],
    'lag_silso_food': [c for c in ['inflation_lag1', 'silso', 'food_price', 'food_anomaly', 'food_production', 'cereal_yield'] if c in model_df],
}
pd.DataFrame([{'model': name, **ols_fit(model_df, 'inflation_shiller', cols)} for name, cols in specs.items()])


## 12. Does inflation mediate the solar-revolution pattern?

This final screen asks whether adding inflation changes the simple SILSO/revolution relationship. If the solar coefficient weakens materially after inflation is included, inflation becomes a plausible mediator or confounder candidate.


In [ ]:
#@title Compare revolution models with and without inflation { display-mode: "form" }
if 'revolutions' in df:
    med_df = df.copy()
    med_df['revolutions_lag1'] = med_df['revolutions'].shift(1)
    specs = {
        'solar_only': ['silso'],
        'solar_plus_inflation': ['silso', 'inflation_shiller'],
        'solar_inflation_lagged_revolutions': ['revolutions_lag1', 'silso', 'inflation_shiller'],
    }
    display(pd.DataFrame([{'model': name, **ols_fit(med_df, 'revolutions', cols)} for name, cols in specs.items()]))
else:
    print('Revolution input not available in this checkout.')


## 13. Falsification summary template

After running the notebook, summarize the evidence against H1 here:

| Test | Result | Does it weaken H1? |
|---|---|---|
| Real SILSO vs shifted/synthetic cycles | TBD | TBD |
| SILSO after lagged inflation and controls | TBD | TBD |
| Historical period stability | TBD | TBD |
| Food/agriculture/energy/rates mechanism candidates | TBD | TBD |
| Revolution mediation-style screen | TBD | TBD |

The preferred conclusion should be skeptical unless several independent tests survive.


### Computed falsification ledger

This cell converts the notebook outputs into an explicit skeptical readout. It is intentionally conservative: if a required table is missing or a test is ambiguous, the verdict is not counted as support for H1.


In [ ]:
#@title Compute falsification ledger { display-mode: "form" }
def safe_float(value):
    try:
        value = float(value)
        return value if np.isfinite(value) else np.nan
    except Exception:
        return np.nan

ledger = []

# 1. One-input ranking: does SILSO stand out before controls?
try:
    silso_rank = inflation_predictors.reset_index(drop=True)
    silso_row = silso_rank[silso_rank["id"] == "climate.solar_activity"].iloc[0]
    rank = int(silso_rank.index[silso_rank["id"] == "climate.solar_activity"][0]) + 1
    ledger.append({
        "test": "one_input_rank",
        "result": f"SILSO rank {rank}; held-out R2={silso_row.heldout_r2:.3f}",
        "weakens_H1": rank != 1 or safe_float(silso_row.heldout_r2) <= 0,
        "keeps_H1_alive": rank == 1 and safe_float(silso_row.heldout_r2) > 0,
    })
except Exception as exc:
    ledger.append({"test": "one_input_rank", "result": f"missing/failed: {exc}", "weakens_H1": True, "keeps_H1_alive": False})

# 2. Incremental controls: does SILSO add value after conventional controls?
try:
    best_delta = safe_float(incremental_silso["delta_r2_from_silso"].max())
    median_delta = safe_float(incremental_silso["delta_r2_from_silso"].median())
    ledger.append({
        "test": "incremental_after_controls",
        "result": f"best delta R2={best_delta:.3f}; median delta R2={median_delta:.3f}",
        "weakens_H1": not (best_delta > 0.02 and median_delta > 0),
        "keeps_H1_alive": best_delta > 0.02 and median_delta > 0,
    })
except Exception as exc:
    ledger.append({"test": "incremental_after_controls", "result": f"missing/failed: {exc}", "weakens_H1": True, "keeps_H1_alive": False})

# 3. Period stability: do correlations keep direction across major regimes?
try:
    signs = np.sign(pd.DataFrame(period_rows)["same_year_r"].dropna())
    stable = len(signs) > 0 and (signs == signs.iloc[0]).all()
    min_abs = safe_float(pd.DataFrame(period_rows)["same_year_r"].abs().min())
    ledger.append({
        "test": "period_stability",
        "result": f"same sign across periods={stable}; minimum |r|={min_abs:.3f}",
        "weakens_H1": not stable or min_abs < 0.15,
        "keeps_H1_alive": stable and min_abs >= 0.15,
    })
except Exception as exc:
    ledger.append({"test": "period_stability", "result": f"missing/failed: {exc}", "weakens_H1": True, "keeps_H1_alive": False})

# 4. Placebo prediction: does real SILSO beat shifted/synthetic cycles?
try:
    placebo_share = safe_float((placebo_pred["r2"] >= real_r2).mean())
    ledger.append({
        "test": "placebo_prediction",
        "result": f"real R2={real_r2:.3f}; placebo share >= real={placebo_share:.3f}",
        "weakens_H1": placebo_share >= 0.10,
        "keeps_H1_alive": placebo_share < 0.10 and real_r2 > 0,
    })
except Exception as exc:
    ledger.append({"test": "placebo_prediction", "result": f"missing/failed: {exc}", "weakens_H1": True, "keeps_H1_alive": False})

# 5. Mechanism candidates: does any plausible candidate correlate with both SILSO and inflation?
try:
    mech = pd.DataFrame(rows)
    plausible = mech[mech["candidate"].isin(["food_price", "food_anomaly", "cereal_yield", "food_production", "temperature", "short_rate", "long_rate", "gdp_growth"])]
    plausible = plausible.assign(min_abs=plausible[["r_silso_candidate", "r_inflation_candidate"]].abs().min(axis=1))
    best_mech = plausible.sort_values("min_abs", ascending=False).head(1)
    best_score = safe_float(best_mech["min_abs"].iloc[0]) if len(best_mech) else np.nan
    best_name = best_mech["candidate"].iloc[0] if len(best_mech) else "none"
    ledger.append({
        "test": "mechanism_candidate_alignment",
        "result": f"best plausible bridge={best_name}; min(|r_silso|, |r_inflation|)={best_score:.3f}",
        "weakens_H1": not (best_score >= 0.20),
        "keeps_H1_alive": best_score >= 0.20,
    })
except Exception as exc:
    ledger.append({"test": "mechanism_candidate_alignment", "result": f"missing/failed: {exc}", "weakens_H1": True, "keeps_H1_alive": False})

falsification_ledger = pd.DataFrame(ledger)
weak_count = int(falsification_ledger["weakens_H1"].sum())
alive_count = int(falsification_ledger["keeps_H1_alive"].sum())
if weak_count >= 3:
    overall = "H1 substantially weakened by current notebook checks"
elif alive_count >= 4:
    overall = "H1 remains alive for follow-up; not confirmed"
else:
    overall = "Mixed/incomplete evidence; do not confirm H1"
print(overall)
display(falsification_ledger)


In [ ]:
#@title Executive conclusion from falsification ledger { display-mode: "form" }
try:
    weak_count = int(falsification_ledger["weakens_H1"].sum())
    alive_count = int(falsification_ledger["keeps_H1_alive"].sum())
    print("Falsification tests weakening H1:", weak_count)
    print("Tests keeping H1 alive:", alive_count)
    if weak_count >= 3:
        print("Conclusion: the solar-inflation hypothesis is substantially weakened by these checks. Treat the original SILSO result as likely artifact/confounding unless new evidence overturns these failures.")
    elif alive_count >= 4 and weak_count <= 1:
        print("Conclusion: H1 survives this screen as an exploratory hypothesis, but it is not confirmed. Next step: preregister controls and test out of sample/non-US inflation data.")
    else:
        print("Conclusion: evidence is mixed or incomplete. Do not claim support for H1; continue adversarial testing.")
except NameError:
    print("Run the computed falsification ledger cell first.")


## 14. Interpretation and next steps

A serious solar-inflation mechanism would need to pass these checks:

1. Real SILSO beats shifted SILSO and generic 11-year sine waves.
2. The lag structure is stable and mechanistically plausible.
3. The signal appears in food or energy prices more clearly than in broad CPI.
4. It survives lagged inflation, rates, war/energy/monetary controls, and period splits.
5. Inflation or food prices reduce the residual SILSO/revolution association in mediation-style screens.

Until then, the inflation result should be described as an exploratory mechanism/confounding candidate.
